# 🚀 Umoja Compute API: gpt-oss:20b via Ollama + Flask + ngrok

**One-click setup** to run `gpt-oss:20b` on a Colab T4 GPU, persist models in Google Drive, and expose a public HTTP API endpoint—turning your ideas into **real-time, actionable AI applications**.

### What you get
- Persistent **14GB model cache** in Google Drive — start instantly without repeated downloads  
- **Ollama server** running entirely on Colab  
- **Flask API** with `/api/generate` endpoint for seamless integration  
- Public URL via **ngrok** — call from Docker, n8n, or any workflow  

> 💡 Tip: Colab sessions are ephemeral. If it stops, re-open the notebook and **Run all** again. Using Drive caching avoids re-downloading the 14GB model each time.  

> 🌟 **Empower your ideas. Deploy AI at the speed of thought.** Umoja Compute makes high-performance AI accessible, modular, and ready for your next breakthrough innovation.

### Quick start
1. Sign up at **[ngrok.com](https://ngrok.com/)** → get your **Auth Token** → paste it into the configuration cell in the notebook  
2. Runtime → *Change runtime type* → **GPU** (T4)  
3. **Run all cells** (Ctrl+F9). Wait for the final cell to print your **Public API URL**  
4. From your machine, call the model:  


```bash
!curl -N -X POST "$Public_URL/v1/completions" -H "Content-Type: application/json" -d '{"model":"gpt-oss:20b","prompt":"Write a single sentence explaining why sunsets are red.","max_tokens":100,"temperature":0.7,"stream":false}'




In [ ]:
# === Configuration with Mode Toggle + Persistence ===
import os, json
import ipywidgets as widgets
from IPython.display import display, clear_output
import uvicorn, requests, httpx

# -----------------
# Defaults
# -----------------
MODEL_NAME = "gpt-oss:20b"   # swap later if needed
USE_DRIVE = True             # set False to use ephemeral storage
NGROK_AUTHTOKEN = ""       # Paste Token between""
ROUTER_PORT = 0              # Router Port (0 = auto-discover)
ROUTER_URL = ""  # only for contributor mode

# -----------------
# Persistence helpers
# -----------------
MODE_FILE = ".umoja_mode"

def save_mode(mode):
    try:
        with open(MODE_FILE, "w") as f:
            json.dump({"mode": mode}, f)
    except Exception as e:
        print("⚠️ Could not save mode:", e)

def load_mode():
    if os.path.exists(MODE_FILE):
        try:
            with open(MODE_FILE, "r") as f:
                return json.load(f).get("mode", "single")
        except Exception:
            return "single"
    return "single"  # default

# -----------------
# Mode application
# -----------------
def apply_mode(mode):
    if mode == "contributor":
        os.environ["ROUTER_URL"] = ROUTER_URL
        os.environ["NODE_ROLE"] = "contributor"
    else:
        os.environ["ROUTER_URL"] = ""
        os.environ["NODE_ROLE"] = "single"

# -----------------
# UI setup
# -----------------
MODE = load_mode()
apply_mode(MODE)

mode_selector = widgets.ToggleButtons(
    options=["contributor", "single"],
    description="Mode:",
    value=MODE,
    button_style=''  # optional styling
)

output = widgets.Output()

def update_mode(change):
    global MODE
    MODE = change['new']
    apply_mode(MODE)
    save_mode(MODE)

    with output:
        clear_output()
        print("🔄 Mode switched to:", MODE)
        print({
            'MODEL_NAME': MODEL_NAME,
            'USE_DRIVE': USE_DRIVE,
            'ROUTER_PORT': ROUTER_PORT,
            'ROUTER_URL': os.environ.get("ROUTER_URL"),
            'NODE_ROLE': os.environ.get("NODE_ROLE")
        })

mode_selector.observe(update_mode, names='value')

display(mode_selector, output)

# Initial print
print({
    'MODEL_NAME': MODEL_NAME,
    'USE_DRIVE': USE_DRIVE,
    'ROUTER_PORT': ROUTER_PORT,
    'NGROK_AUTHTOKEN_SET': bool(NGROK_AUTHTOKEN),
    'ROUTER_URL': os.environ.get("ROUTER_URL"),
    'NODE_ROLE': os.environ.get("NODE_ROLE"),
    'STARTUP_MODE': MODE
})


ToggleButtons(description='Mode:', index=1, options=('contributor', 'single'), value='single')

Output()

{'MODEL_NAME': 'gpt-oss:20b', 'USE_DRIVE': True, 'ROUTER_PORT': 0, 'NGROK_AUTHTOKEN_SET': True, 'ROUTER_URL': '', 'NODE_ROLE': 'single', 'STARTUP_MODE': 'single'}


In [2]:
# Check GPU availability
!nvidia-smi || true

Mon Mar  2 06:43:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Mount Google Drive (persistent storage for models) if enabled
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    %env OLLAMA_MODELS=/content/drive/MyDrive/.ollama/models
else:
    %env OLLAMA_MODELS=/root/.ollama/models

# Ensure the model directory exists
import os
os.makedirs(os.environ.get('OLLAMA_MODELS', '/root/.ollama/models'), exist_ok=True)
print('OLLAMA_MODELS ->', os.environ.get('OLLAMA_MODELS'))

Mounted at /content/drive
env: OLLAMA_MODELS=/content/drive/MyDrive/.ollama/models
OLLAMA_MODELS -> /content/drive/MyDrive/.ollama/models


In [4]:
#Ollama Requisite
!apt-get update
!apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,385 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,302 kB]
Hit:13 https://ppa.launchpadcontent.net

In [5]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh
!ollama --version

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [6]:
# Start Ollama server in the background
import subprocess, time, os
ollama_proc = subprocess.Popen(["bash", "-lc", "OLLAMA_MODELS='$OLLAMA_MODELS' ollama serve"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
time.sleep(3)
print("Ollama server started (background).")

Ollama server started (background).


In [7]:
# Pull the model (first time will download ~14GB for gpt-oss:20b)
import os
print('Using model store at:', os.environ.get('OLLAMA_MODELS'))
!OLLAMA_MODELS=$OLLAMA_MODELS ollama pull gpt-oss:20b

Using model store at: /content/drive/MyDrive/.ollama/models



In [8]:
!ollama run gpt-oss:20b "Hello?"

Thinking...
We need to respond to user greeting. We need to follow guidelines: friendly, helpful.
...done thinking.

Hello! 👋 How can I help you today?



In [9]:
# Install Python libraries for API + tunneling
!pip install flask pyngrok requests --quiet

from pyngrok import ngrok

# Hardcoded ngrok authtoken
NGROK_AUTHTOKEN = NGROK_AUTHTOKEN

# Set the token
ngrok.set_auth_token(NGROK_AUTHTOKEN)

print('Libraries installed. ngrok authtoken set:', bool(NGROK_AUTHTOKEN))

Libraries installed. ngrok authtoken set: True


In [10]:
# === Flask API server (OpenAI-compatible wrapper around Ollama) ===
import threading, requests, json, os, time, uuid
from flask import Flask, request, jsonify, Response, stream_with_context

FLASK_PORT = 5000
OLLAMA_URL = "http://127.0.0.1:11434"

app = Flask(__name__)

# ---------------------------
# Health check
# ---------------------------
@app.get('/health')
def health():
    try:
        r = requests.get(OLLAMA_URL + "/api/tags", timeout=10)
        r.raise_for_status()
        models = [m['name'] for m in r.json().get('models', [])]
        return {"status": "healthy", "models": models}, 200
    except Exception as e:
        return {"status": "unhealthy", "error": str(e)}, 503


# ---------------------------
# /v1/models
# ---------------------------
@app.get('/v1/models')
def list_models():
    try:
        r = requests.get(OLLAMA_URL + "/api/tags", timeout=10)
        r.raise_for_status()
        data = r.json()
        models = [
            {
                "id": m["name"],
                "object": "model",
                "created": int(time.time()),
                "owned_by": "ollama",
                "permission": []
            }
            for m in data.get("models", [])
        ]
        return jsonify({"object": "list", "data": models})
    except Exception as e:
        return jsonify({"error": str(e)}), 500


# ===========================
# Utility: Stream Wrapper
# ===========================
def stream_from_ollama(payload):
    """Streams response from Ollama /api/generate endpoint in OpenAI-style SSE format."""
    with requests.post(
        OLLAMA_URL + "/api/generate",
        json=payload,
        stream=True,
        timeout=(60, None)  # connect timeout + infinite read timeout
    ) as r:
        for chunk in r.iter_lines():
            if not chunk:
                continue
            try:
                j = json.loads(chunk)
                data = {
                    "id": f"cmpl-{uuid.uuid4().hex}",
                    "object": "chat.completion.chunk",
                    "created": int(time.time()),
                    "model": j.get("model", payload["model"]),
                    "choices": [{
                        "delta": {"content": j.get("response", "")},
                        "index": 0,
                        "finish_reason": None
                    }]
                }
                yield f"data: {json.dumps(data)}\n\n".encode()
            except Exception as e:
                print(f"⚠️ Bad chunk: {chunk} | {e}")

        # Send the [DONE] event when streaming is complete
        yield b"data: [DONE]\n\n"



# ===========================
# Chat Completions (OpenAI-style)
# ===========================
@app.route("/v1/chat/completions", methods=["POST"])
def chat_completions():
    body = request.json
    prompt = body.get("prompt")  # your router already builds prompt from messages
    payload = {
        "model": body.get("model", "gpt-oss:20b"),
        "prompt": prompt,
        "stream": body.get("stream", False)
    }

    if payload["stream"]:
        return Response(stream_from_ollama(payload), content_type="text/event-stream")

    # Non-stream mode
    r = requests.post(OLLAMA_URL + "/api/generate", json=payload, timeout=60)
    r.raise_for_status()
    j = r.json()
    return jsonify({"model": j.get("model", payload["model"]), "response": j.get("response", "")})


# ===========================
# Text Completions
# ===========================
@app.route("/v1/completions", methods=["POST"])
def completions():
    body = request.json
    payload = {
        "model": body.get("model", "gpt-oss:20b"),
        "prompt": body.get("prompt", ""),
        "stream": body.get("stream", False)
    }

    if payload["stream"]:
        return Response(stream_from_ollama(payload), content_type="text/event-stream")

    # Non-stream mode
    r = requests.post(OLLAMA_URL + "/api/generate", json=payload, timeout=60)
    r.raise_for_status()
    j = r.json()
    return jsonify({"model": j.get("model", payload["model"]), "response": j.get("response", "")})


# ---------------------------
# /v1/embeddings
# ---------------------------
@app.post('/v1/embeddings')
def embeddings():
    data = request.get_json(force=True)
    input_text = data.get("input", "")
    model = data.get("model", "gpt-oss:embedding")

    payload = {"model": model, "input": input_text}
    r = requests.post(OLLAMA_URL + "/api/embeddings", json=payload, timeout=600)
    r.raise_for_status()
    result = r.json()

    return jsonify({
        "object": "list",
        "data": [{"object": "embedding", "embedding": result.get("embedding", []), "index": 0}],
        "model": model,
        "usage": {"prompt_tokens": 0, "total_tokens": 0}
    })


# ---------------------------
# Launch server with infinite readiness loop
# ---------------------------
def wait_for_ollama():
    print("⏳ Waiting for Ollama to become ready...")
    while True:
        try:
            r = requests.get(OLLAMA_URL + "/api/tags", timeout=5)
            if r.status_code == 200:
                print("✅ Ollama is ready.")
                return True
        except:
            pass
        print("...Ollama not ready yet, retrying in 2s")
        time.sleep(2)


def run_app():
    app.run(host='0.0.0.0', port=FLASK_PORT)


if wait_for_ollama():
    thread = threading.Thread(target=run_app, daemon=True)
    thread.start()


⏳ Waiting for Ollama to become ready...
✅ Ollama is ready.


In [11]:
import requests
import time

FLASK_URL = "http://127.0.0.1:5000"  # change if using ngrok/public URL

# Wait a few seconds for the server to start
time.sleep(3)

try:
    response = requests.get(f"{FLASK_URL}/health", timeout=5)
    if response.status_code == 200:
        print("✅ Server is running!")
        print("Response:", response.json())
    else:
        print("⚠️ Server responded with status:", response.status_code)
        print("Response:", response.text)
except requests.RequestException as e:
    print("❌ Could not connect to server:", e)


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [02/Mar/2026 06:50:23] "GET /health HTTP/1.1" 200 -


✅ Server is running!
Response: {'models': ['gpt-oss:20b'], 'status': 'healthy'}


In [12]:
!pip install fastapi uvicorn nest_asyncio pyngrok sse-starlette pydantic requests

In [13]:
# === Umoja API Engine — OpenAI-compatible Gateway (Dashboard + Router Ready) ===
# Requirements: fastapi, uvicorn, pyngrok, pydantic, nest_asyncio, requests, aiohttp, httpx
import os, sys, time, threading, json, uuid, requests, aiohttp, uvicorn, nest_asyncio, logging, asyncio
from typing import Dict
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse, HTMLResponse
from pyngrok import ngrok

# -------------------
# Basic logging
# -------------------
logging.basicConfig(level=logging.INFO)

# -------------------
# Config
# -------------------
API_PORT = int(os.environ.get("API_PORT", 8000))
MODEL_NAME = os.environ.get("MODEL_NAME", "gpt-oss:20b")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434")
HEARTBEAT_INTERVAL = int(os.environ.get("HEARTBEAT_INTERVAL", 10))
ROUTER_PORT = int(os.environ.get("ROUTER_PORT", 0))
NODE_ROLE = os.environ.get("NODE_ROLE", "single")  # "single" | "contributor"

# -------------------
# Node / Metrics State
# -------------------
node_id = f"engine-{uuid.uuid4().hex[:8]}"
_metrics = {
    "total_requests": 0,
    "total_prompt_tokens": 0,
    "total_completion_tokens": 0,
    "history": [],   # rolling token usage history
    "latencies": [], # optional latency samples
}
# node discovery / bookkeeping used by dashboard / status
_nodes: Dict[str, dict] = {}
_nodes_lock = asyncio.Lock()
_node_stats: Dict[str, dict] = {}
_logs = []
_log_queue: asyncio.Queue = asyncio.Queue()
MAX_LOGS = 200
NODE_TTL = 30

# -------------------
# Helpers (router discovery / public url)
# -------------------
def get_router_url():
    tunnels = ngrok.get_tunnels()
    for t in tunnels:
        try:
            addr = t.config.get("addr", "")
        except Exception:
            addr = ""
        if "http" in t.public_url:
            if ROUTER_PORT and str(ROUTER_PORT) in addr:
                return t.public_url
            if not ROUTER_PORT:
                return t.public_url
    raise RuntimeError("❌ Router tunnel not found — start Router Cell first")

if NODE_ROLE == "contributor":
    ROUTER_URL = os.environ.get("ROUTER_URL") or get_router_url()
    print(f"🔗 Discovered Router URL: {ROUTER_URL}")
else:
    ROUTER_URL = os.environ.get("ROUTER_URL", "")
    print("🟢 Single mode — skipping router discovery")

# Always start tunnel for API_PORT
public_url = ngrok.connect(API_PORT, "http").public_url
print(f"🔗 Engine public URL: {public_url}")

# -------------------
# Logging & Metrics helpers
# -------------------
def log_event(msg: str, level: str = "info"):
    entry = {
        "t": int(time.time()),
        "msg": msg,
        "level": level
    }
    _logs.append(entry)
    if len(_logs) > MAX_LOGS:
        _logs.pop(0)
    try:
        _log_queue.put_nowait(entry)
    except Exception:
        pass
    if level == "info":
        logging.info(msg)
    elif level == "warn":
        logging.warning(msg)
    elif level == "crit":
        logging.error(msg)
    else:
        logging.debug(msg)

def _update_metrics(usage: dict, latency: float = None):
    _metrics["total_requests"] += 1
    _metrics["total_prompt_tokens"] += usage.get("prompt_tokens", 0)
    _metrics["total_completion_tokens"] += usage.get("completion_tokens", 0)
    total_tokens = usage.get("prompt_tokens", 0) + usage.get("completion_tokens", 0)
    _metrics["history"].append({"t": int(time.time()), "tokens": total_tokens})
    if len(_metrics["history"]) > 50:
        _metrics["history"].pop(0)
    if latency is not None:
        _metrics["latencies"].append(latency)
        if len(_metrics["latencies"]) > 100:
            _metrics["latencies"].pop(0)

def _track_node_request(nid: str):
    now = int(time.time())
    if nid not in _node_stats:
        _node_stats[nid] = {"requests": 0, "timestamps": []}
    _node_stats[nid]["requests"] += 1
    _node_stats[nid]["timestamps"].append(now)
    _node_stats[nid]["timestamps"] = _node_stats[nid]["timestamps"][-100:]

def _active_nodes() -> Dict[str, Dict]:
    now = time.time()
    return {
        nid: info for nid, info in _nodes.items()
        if info.get("last_seen") and (now - info["last_seen"] < NODE_TTL)
    }

# -------------------
# FastAPI App
# -------------------
app = FastAPI(title="Umoja API Engine")

@app.get("/health")
def health():
    # update our local node last seen in _nodes for discovery
    try:
        _nodes[node_id] = {
            "id": node_id,
            "url": public_url,
            "model": MODEL_NAME,
            "last_seen": time.time(),
            "is_active": True,
            "last_probe_status": True,
        }
    except Exception:
        pass
    return {
        "status": "ok",
        "router_url": ROUTER_URL,
        "model": MODEL_NAME,
        "node_role": NODE_ROLE,
        "time": int(time.time())
    }

@app.get("/status")
def status():
    latencies = _metrics.get("latencies", [])
    avg_latency = round(sum(latencies) / len(latencies), 2) if latencies else 0
    safe_history = _metrics.get("history", [])[-50:]

    all_nodes = dict(_active_nodes())
    # always include local node info
    all_nodes[node_id] = {
        "id": node_id,
        "url": public_url,
        "model": MODEL_NAME,
        "last_seen": time.time(),
        "is_active": True,
        "last_probe_status": True,
    }

    return {
        "node_id": node_id,
        "node_role": NODE_ROLE,
        "public_url": public_url,
        "model": MODEL_NAME,
        "active_nodes": len(all_nodes),
        "all_nodes": all_nodes,
        "metrics": {**_metrics, "history": safe_history},
        "node_stats": _node_stats,
        "requests_served": _metrics.get("total_requests", 0),
        "average_latency_ms": avg_latency,
        "time": int(time.time())
    }

@app.get("/v1/models")
def list_models():
    return {
        "object": "list",
        "data": [{"id": MODEL_NAME, "object": "model", "owned_by": node_id}]
    }

# -------------------
# Logs Endpoint (SSE)
# -------------------
@app.get("/logs")
async def stream_logs():
    async def event_gen():
        # yield existing logs
        for entry in _logs:
            yield f"data: {json.dumps(entry)}\n\n"
        while True:
            entry = await _log_queue.get()
            yield f"data: {json.dumps(entry)}\n\n"
    return StreamingResponse(event_gen(), media_type="text/event-stream")

# -------------------
# Ollama Wrappers
# -------------------
async def ollama_chat_stream(model: str, messages: list):
    url = f"{OLLAMA_URL}/api/chat"
    payload = {"model": model, "messages": messages, "stream": True}
    prompt_tokens = sum(len(m.get("content", "").split()) for m in messages)
    usage_acc = {"prompt_tokens": prompt_tokens, "completion_tokens": 0}

    async with aiohttp.ClientSession() as session:
        async with session.post(url, json=payload) as resp:
            async for line in resp.content:
                line = line.strip()
                if not line: continue
                try:
                    event = json.loads(line.decode("utf-8"))
                except Exception:
                    continue
                if "message" in event:
                    msg = event["message"]
                    if msg.get("role") == "assistant" and msg.get("content"):
                        usage_acc["completion_tokens"] += len(msg["content"].split())
                        yield {"message": {"role": "assistant", "content": msg["content"]}}
                if event.get("done", False):
                    # update metrics at stream end
                    _update_metrics({"prompt_tokens": usage_acc["prompt_tokens"], "completion_tokens": usage_acc["completion_tokens"]})
                    yield {"done": True}
                    break

async def ollama_chat_wrapper_nonstream(model: str, messages: list):
    url = f"{OLLAMA_URL}/api/chat"
    payload = {"model": model, "messages": messages, "stream": False}
    async with aiohttp.ClientSession() as session:
        try:
            async with session.post(url, json=payload) as resp:
                data = await resp.json()
        except Exception as e:
            logging.exception("❌ Ollama request failed")
            prompt_tokens = sum(len(m.get("content", "").split()) for m in messages)
            # still count this attempt as a request
            _update_metrics({"prompt_tokens": prompt_tokens, "completion_tokens": 0})
            return {
                "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
                "object": "chat.completion",
                "created": int(time.time()),
                "model": model,
                "choices": [{"index": 0, "message": {"role": "assistant", "content": "[ENGINE ERROR: Failed]"},
                             "finish_reason": "stop"}],
                "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": 0, "total_tokens": prompt_tokens},
                "raw": {}
            }

    content = ""
    if "message" in data:
        content = data["message"].get("content", "")
    elif "choices" in data and data["choices"]:
        content = data["choices"][0].get("message", {}).get("content", "")
    elif "response" in data:
        r = data["response"]
        if isinstance(r, list) and r:
            content = r[0].get("content", "")
        elif isinstance(r, dict):
            content = r.get("content", "")

    token_count = len(content.split())
    prompt_tokens = sum(len(m.get("content", "").split()) for m in messages)

    _update_metrics({"prompt_tokens": prompt_tokens, "completion_tokens": token_count})
    return {
        "id": f"chatcmpl-{uuid.uuid4().hex[:12]}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": model,
        "choices": [{"index": 0, "message": {"role": "assistant", "content": content},
                     "finish_reason": data.get("done_reason", "stop")}],
        "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": token_count, "total_tokens": prompt_tokens + token_count},
        "raw": data
    }

async def ollama_chat_wrapper_stream(model: str, messages: list):
    async for event in ollama_chat_stream(model, messages):
        yield event

# -------------------
# OpenAI-compatible Endpoints
# (we keep your existing endpoints but add logging/metrics hooks)
# -------------------
@app.post("/v1/completions")
async def completions(request: Request):
    body = await request.json()
    prompt, stream = body.get("prompt", ""), body.get("stream", False)
    cmpl_id, created = f"cmpl-{uuid.uuid4().hex[:12]}", int(time.time())
    prompt_tokens, completion_tokens = len(prompt.split()), 0
    start_t = time.time()

    if stream:
        async def event_generator():
            nonlocal completion_tokens
            async for event in ollama_chat_wrapper_stream(MODEL_NAME, [{"role": "user", "content": prompt}]):
                if "message" in event:
                    content = event["message"]["content"]
                    completion_tokens += len(content.split())
                    chunk = {"id": cmpl_id, "object": "completion.chunk", "created": created,
                             "model": MODEL_NAME, "choices": [{"index": 0, "text": content, "finish_reason": None}]}
                    yield f"data: {json.dumps(chunk)}\n\n"
                if event.get("done"):
                    latency_ms = (time.time() - start_t) * 1000
                    usage = {"prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens}
                    # _update_metrics already called inside stream wrapper; ensure node request tracked & log
                    _track_node_request(node_id)
                    log_event(f"Streamed completion served in {latency_ms:.1f}ms", "info")
                    usage_chunk = {"id": cmpl_id, "object": "completion.chunk", "created": created,
                                   "model": MODEL_NAME, "choices": [],
                                   "usage": {"prompt_tokens": prompt_tokens,
                                             "completion_tokens": completion_tokens,
                                             "total_tokens": prompt_tokens+completion_tokens}}
                    yield f"data: {json.dumps(usage_chunk)}\n\n"
                    yield "data: [DONE]\n\n"
                    break
        return StreamingResponse(event_generator(), media_type="text/event-stream")

    # non-stream path
    resp = await ollama_chat_wrapper_nonstream(MODEL_NAME, [{"role": "user", "content": prompt}])
    content, completion_tokens = resp["choices"][0]["message"]["content"], resp["usage"]["completion_tokens"]
    latency_ms = (time.time() - start_t) * 1000
    # ensure we track node and log
    _track_node_request(node_id)
    log_event(f"Completion served in {latency_ms:.1f}ms", "info")

    return JSONResponse({
        "id": cmpl_id, "object": "completion", "created": created, "model": MODEL_NAME,
        "choices": [{"index": 0, "text": content, "finish_reason": "stop"}],
        "usage": {"prompt_tokens": prompt_tokens, "completion_tokens": completion_tokens, "total_tokens": prompt_tokens+completion_tokens}
    })

@app.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    model, messages, stream = body.get("model", MODEL_NAME), body.get("messages", []), body.get("stream", False)
    chat_id, created = f"chatcmpl-{uuid.uuid4().hex[:12]}", int(time.time())
    prompt_tokens, completion_tokens = sum(len(m.get("content","").split()) for m in messages), 0
    start_t = time.time()

    if stream:
        async def event_generator():
            nonlocal completion_tokens
            async for event in ollama_chat_wrapper_stream(model, messages):
                if "message" in event:
                    content = event["message"]["content"]
                    completion_tokens += len(content.split())
                    chunk = {"id": chat_id, "object": "chat.completion.chunk", "created": created,
                             "model": model, "choices": [{"index": 0, "delta": {"content": content}, "finish_reason": None}]}
                    yield f"data: {json.dumps(chunk)}\n\n"
                if event.get("done"):
                    latency_ms = (time.time() - start_t) * 1000
                    _track_node_request(node_id)
                    log_event(f"Streamed chat completion served in {latency_ms:.1f}ms", "info")
                    stop_chunk = {"id": chat_id, "object": "chat.completion.chunk", "created": created,
                                  "model": model, "choices": [{"index":0,"delta":{},"finish_reason":"stop"}]}
                    usage_chunk = {"id": chat_id, "object": "chat.completion.chunk", "created": created,
                                   "model": model, "choices": [],
                                   "usage": {"prompt_tokens": prompt_tokens,
                                             "completion_tokens": completion_tokens,
                                             "total_tokens": prompt_tokens+completion_tokens}}
                    yield f"data: {json.dumps(stop_chunk)}\n\n"
                    yield f"data: {json.dumps(usage_chunk)}\n\n"
                    yield "data: [DONE]\n\n"
                    break
        return StreamingResponse(event_generator(), media_type="text/event-stream")

    resp = await ollama_chat_wrapper_nonstream(model, messages)
    final_text, completion_tokens = resp["choices"][0]["message"]["content"], resp["usage"]["completion_tokens"]
    latency_ms = (time.time() - start_t) * 1000
    _track_node_request(node_id)
    log_event(f"Chat completion served in {latency_ms:.1f}ms", "info")
    return JSONResponse({
        "id": chat_id, "object": "chat.completion", "created": created, "model": model,
        "choices": [{"index":0,"message":{"role":"assistant","content":final_text},"finish_reason":"stop"}],
        "usage":{"prompt_tokens":prompt_tokens,"completion_tokens":completion_tokens,"total_tokens":prompt_tokens+completion_tokens}
    })

# =============================
# Dashboard Route (Revamped + Logs)
# =============================
@app.get("/", response_class=HTMLResponse)
async def dashboard():
    html = r"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8" />
        <meta name="viewport" content="width=device-width, initial-scale=1.0" />
        <title>🚦 Umoja Router - Dashboard</title>
        <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
        <style>
            :root {
                --bg-dark: #0f172a;
                --bg-card: #1e293b;
                --text: #f1f5f9;
                --accent: #3b82f6;
                --success: #10b981;
                --danger: #ef4444;
                --warn: #facc15;
                --shadow: 0 4px 12px rgba(0,0,0,0.45);
            }
            body { font-family: system-ui, Arial, sans-serif; background: var(--bg-dark); color: var(--text); margin: 0; padding: 0; }
            nav { background: var(--bg-card); padding: 1rem; text-align: center; box-shadow: var(--shadow); position: sticky; top: 0; z-index: 10; }
            nav a { margin: 0 15px; color: var(--text); text-decoration: none; font-weight: 600; transition: color 0.2s ease; }
            nav a:hover { color: var(--accent); }

            .container { max-width: 1400px; margin: auto; padding: 20px; display: grid; grid-template-columns: repeat(auto-fit, minmax(300px, 1fr)); gap: 20px; }
            .card { background: var(--bg-card); border-radius: 16px; padding: 20px; box-shadow: var(--shadow); display: flex; flex-direction: column; justify-content: center; }
            .kpi { font-size: 2rem; font-weight: bold; margin: 10px 0; display: flex; align-items: center; gap: 8px; }
            .trend { font-size: 1rem; font-weight: 600; display: inline-flex; align-items: center; gap: 2px; }
            .trend.up { color: var(--success); }
            .trend.down { color: var(--danger); }

            .chart-card { grid-column: span 2; }
            .status-badge { padding: 4px 8px; border-radius: 8px; font-weight: bold; font-size: 0.9rem; margin-top: 6px; display: inline-block; transition: all 0.3s ease; }
            .status-ok { background: var(--success); color: black; }
            .status-warn { background: var(--warn); color: black; }
            .status-crit { background: var(--danger); }

            .logs-card { grid-column: span 2; background: #111827; font-family: monospace; font-size: 0.85rem; height: 200px; overflow-y: auto; border-radius: 12px; padding: 10px; box-shadow: inset 0 0 8px rgba(0,0,0,0.4); }
            .log-line { padding: 2px 0; color: #cbd5e1; border-bottom: 1px solid rgba(255,255,255,0.02); }
            .log-line.warn { color: var(--warn); }
            .log-line.crit { color: var(--danger); }

            canvas { width: 100% !important; height: auto !important; }
            .refresh-note { text-align: center; font-size: 0.85rem; opacity: 0.7; margin-top: 8px; }
            #loading { text-align: center; margin-top: 8px; display: none; color: #facc15; font-weight: bold; }

            @keyframes fadeIn { from { opacity: 0; transform: translateY(8px); } to { opacity: 1; transform: translateY(0); } }
        </style>
    </head>
    <body>
        <nav>
            <a href="/">Dashboard</a>
            <a href="/playground">Playground</a>
            <a href="/about">About</a>
            <a href="/vc-pitch">VC Pitch</a>
        </nav>

        <div class="container">
            <div class="card"><h2>📈 Total Requests</h2><div class="kpi"><span id="totalRequests">0</span> <span class="trend" id="trendRequests">–</span></div></div>
            <div class="card"><h2>🔤 Total Tokens</h2><div class="kpi"><span id="totalTokens">0</span> <span class="trend" id="trendTokens">–</span></div></div>
            <div class="card"><h2>⏱ Avg Latency</h2><div class="kpi"><span id="avgLatency">0 ms</span> <span class="trend" id="trendLatency">–</span></div><canvas id="latencySparkline" height="40"></canvas></div>
            <div class="card"><h2>⚡ Active Nodes</h2><div class="kpi"><span id="nodeCount">Loading...</span></div><div id="systemHealth" class="status-badge status-ok">Checking...</div></div>

            <div class="card chart-card"><h2>📊 Requests Over Time</h2><canvas id="requestsChart"></canvas><div id="loading">⏳ Loading metrics...</div><p class="refresh-note">Auto-refreshes every 10s</p></div>

            <div class="card logs-card"><h2>🖥 Live Logs</h2><div id="liveLogs">No live logs endpoint — showing activity feed</div></div>
        </div>

        <script>
            // Charts and state
            let requestsChart = null, latencyChart = null;
            let latencyHistory = [];
            let prevStatus = null;             // for synthesizing logs & diffs
            let clientRequestHistory = [];     // fallback when metrics.history is absent
            const MAX_LOG_LINES = 200;
            let logLines = [];

            function pushLog(text, level = "info") {
                const logDiv = document.getElementById('liveLogs');
                const line = document.createElement('div');
                line.className = 'log-line' + (level === 'warn' ? ' warn' : (level === 'crit' ? ' crit' : ''));
                const now = new Date().toLocaleTimeString();
                line.textContent = `[${now}] ${text}`;
                logDiv.appendChild(line);
                logLines.push({ text: line.textContent, level });
                if (logLines.length > MAX_LOG_LINES) {
                    // remove top-most DOM nodes to keep it light
                    logLines.shift();
                    if (logDiv.firstChild) logDiv.removeChild(logDiv.firstChild);
                }
                logDiv.scrollTop = logDiv.scrollHeight;
            }

            async function fetchStatus() {
                try {
                    document.getElementById('loading').style.display = 'block';
                    const res = await fetch('/status');
                    document.getElementById('loading').style.display = 'none';
                    return await res.json();
                } catch (e) {
                    console.error('Failed to fetch /status', e);
                    document.getElementById('loading').style.display = 'none';
                    return { metrics: { history: [] }, requests_served: 0, average_latency_ms: 0, active_nodes: 0 };
                }
            }

            function parseHistoryEntries(history) {
                // history items can be many shapes. Try to extract time (seconds/ms) and a numeric value.
                const items = [];
                for (const h of history) {
                    // time candidates
                    const timeCandidates = [h.t, h.timestamp, h.ts, h.time, h.time_ms, h.last_seen, h.seen];
                    let timeVal = null;
                    for (const c of timeCandidates) { if (c !== undefined && c !== null) { timeVal = c; break; } }
                    // normalize to ms
                    let timeMs = null;
                    if (timeVal !== null) {
                        // if looks like a number string, cast
                        if (typeof timeVal === 'string' && /^\d+$/.test(timeVal)) timeVal = Number(timeVal);
                        if (typeof timeVal === 'number') {
                            // heuristics: > 1e12 treat as ms, > 1e9 treat as seconds
                            timeMs = timeVal > 1e12 ? timeVal : (timeVal > 1e9 ? timeVal * 1000 : timeVal * 1000);
                        }
                    }
                    // value candidates
                    const valCandidates = [h.tokens, h.count, h.value, h.requests, h.requests_count, h.requests_served, h.tokens_processed];
                    let val = null;
                    for (const c of valCandidates) { if (c !== undefined && c !== null) { val = Number(c); break; } }
                    // fallback: if item has numeric properties, try first numeric prop
                    if (val === null) {
                        for (const k in h) {
                            if (Object.prototype.hasOwnProperty.call(h, k) && typeof h[k] === 'number') { val = h[k]; break; }
                        }
                    }
                    if (timeMs === null) {
                        // can't parse time — use current time offset by index (still useful)
                        timeMs = Date.now();
                    }
                    if (val === null) val = 0;
                    items.push({ timeMs, val });
                }
                return items;
            }

            function buildRequestsSeriesFromMetrics(metrics) {
                // prefer server-provided history array when it's populated
                if (metrics && Array.isArray(metrics.history) && metrics.history.length) {
                    const items = parseHistoryEntries(metrics.history);
                    // sort by time just in case
                    items.sort((a,b)=>a.timeMs - b.timeMs);
                    const labels = items.map(it => new Date(it.timeMs).toLocaleTimeString());
                    const values = items.map(it => it.val);
                    return { labels, values };
                }
                // fallback: use client-side total_requests trend to compute deltas
                const now = Date.now();
                const total = metrics.total_requests || metrics.requests_served || 0;
                clientRequestHistory.push({ timeMs: now, total });
                if (clientRequestHistory.length > 60) clientRequestHistory.shift(); // keep one-minute-ish buffer
                // compute deltas (requests per interval)
                const labels = [];
                const values = [];
                for (let i = 1; i < clientRequestHistory.length; i++) {
                    labels.push(new Date(clientRequestHistory[i].timeMs).toLocaleTimeString());
                    const delta = clientRequestHistory[i].total - clientRequestHistory[i-1].total;
                    values.push(delta >= 0 ? delta : 0);
                }
                return { labels, values };
            }

            function updateTrend(id,newValue,oldValue,unit=""){ const el=document.getElementById(id); if (oldValue===undefined) return el.textContent="–"; if (newValue>oldValue){ el.textContent=`▲ ${(newValue-oldValue)}${unit}`; el.className="trend up"; } else if (newValue<oldValue){ el.textContent=`▼ ${(oldValue-newValue)}${unit}`; el.className="trend down"; } else { el.textContent="–"; el.className="trend"; } }

            function synthesizeLogs(prev, curr) {
                // create readable lines for key events: new requests, latency change, node up/down, node discovery
                if (!prev) {
                    pushLog('Dashboard connected');
                    if (curr && curr.all_nodes) {
                        for (const [id, node] of Object.entries(curr.all_nodes)) {
                            pushLog(`Node discovered: ${id} (${node.url || 'local'})`);
                        }
                    }
                    return;
                }
                // new requests
                const prevReq = prev.requests_served || prev.metrics && prev.metrics.total_requests || 0;
                const currReq = curr.requests_served || curr.metrics && curr.metrics.total_requests || 0;
                if (currReq > prevReq) {
                    pushLog(`${currReq - prevReq} new request(s) processed`);
                }
                // latency change (significant)
                const prevLat = prev.average_latency_ms ?? 0;
                const currLat = curr.average_latency_ms ?? 0;
                if (currLat !== prevLat) {
                    const diff = Math.round(currLat - prevLat);
                    const lvl = Math.abs(diff) > 200 ? 'crit' : (Math.abs(diff) > 50 ? 'warn' : 'info');
                    pushLog(`Avg latency ${currLat} ms (${diff >= 0 ? '+' : ''}${diff} ms)`, lvl);
                }
                // nodes changes
                const prevNodes = (prev.all_nodes) ? { ...prev.all_nodes } : {};
                const currNodes = (curr.all_nodes) ? { ...curr.all_nodes } : {};
                // union of keys
                const nodeIds = new Set([...Object.keys(prevNodes), ...Object.keys(currNodes)]);
                for (const id of nodeIds) {
                    const p = prevNodes[id];
                    const c = currNodes[id];
                    if (!p && c) {
                        pushLog(`🆕 Node added: ${id}`, 'info');
                    } else if (p && !c) {
                        pushLog(`❌ Node removed: ${id}`, 'warn');
                    } else if (p && c) {
                        // check probe status changes (true/false/null)
                        const pStatus = p.last_probe_status;
                        const cStatus = c.last_probe_status;
                        if (pStatus !== cStatus) {
                            if (cStatus === true) pushLog(`🔄 Node recovered: ${id}`, 'info');
                            else if (cStatus === false) pushLog(`⚠️ Node failed: ${id}`, 'warn');
                        }
                        // check active flag
                        if ((p.is_active || false) !== (c.is_active || false)) {
                            if (c.is_active) pushLog(`🔵 Node active: ${id}`, 'info'); else pushLog(`⚪ Node inactive: ${id}`, 'warn');
                        }
                    }
                }
            }

            async function renderDashboard() {
                const status = await fetchStatus();
                // metrics fallback
                const metrics = status.metrics || (status.metrics === undefined ? { history: [] } : {});
                const totalTokens = (metrics.total_prompt_tokens || 0) + (metrics.total_completion_tokens || 0);

                // update trends
                updateTrend('trendRequests', metrics.total_requests || status.requests_served || 0, (prevStatus && (prevStatus.metrics && prevStatus.metrics.total_requests || prevStatus.requests_served)) );
                updateTrend('trendTokens', totalTokens, prevStatus && ((prevStatus.metrics && prevStatus.metrics.total_prompt_tokens || 0) + (prevStatus.metrics && prevStatus.metrics.total_completion_tokens || 0)));
                updateTrend('trendLatency', status.average_latency_ms || 0, prevStatus && (prevStatus.average_latency_ms || 0), 'ms');

                document.getElementById('totalRequests').innerText = (metrics.total_requests || status.requests_served || 0).toLocaleString();
                document.getElementById('totalTokens').innerText = totalTokens.toLocaleString();
                document.getElementById('avgLatency').innerText = (status.average_latency_ms || 0) + ' ms';
                document.getElementById('nodeCount').innerText = (status.active_nodes || 0) + ' active nodes';

                // system health
                const healthBadge = document.getElementById('systemHealth');
                if (status.all_nodes) {
                    const unhealthy = Object.values(status.all_nodes).some(n => n.last_probe_status === false);
                    healthBadge.textContent = unhealthy ? 'Degraded' : 'Healthy';
                    healthBadge.className = 'status-badge ' + (unhealthy ? 'status-warn' : 'status-ok');
                } else {
                    healthBadge.textContent = 'Unknown';
                    healthBadge.className = 'status-badge status-crit';
                }

                // latency sparkline
                latencyHistory.push(status.average_latency_ms || 0);
                if (latencyHistory.length > 20) latencyHistory.shift();
                if (!latencyChart) {
                    latencyChart = new Chart(document.getElementById('latencySparkline'), {
                        type: 'line',
                        data: { labels: latencyHistory.map((_,i)=>i), datasets: [{ data: latencyHistory, borderColor: '#3b82f6', fill: false, tension: 0.3, pointRadius: 0 }] },
                        options: { animation: false, scales: { x: { display: false }, y: { display: false } }, plugins: { legend: { display: false } } }
                    });
                } else {
                    latencyChart.data.datasets[0].data = latencyHistory;
                    latencyChart.update('none');
                }

                // requests over time: handle multiple history shapes; fallback to client-side delta series
                const series = buildRequestsSeriesFromMetrics(metrics);
                const labels = series.labels.length ? series.labels : ['-'];
                const data = series.values.length ? series.values : [0];

                if (!requestsChart) {
                    requestsChart = new Chart(document.getElementById('requestsChart'), {
                        type: 'line',
                        data: { labels, datasets: [{ label: 'Requests', data, borderColor: '#10b981', tension: 0.3, fill: false }] },
                        options: { animation: false, scales: { y: { beginAtZero: true } } }
                    });
                } else {
                    requestsChart.data.labels = labels;
                    requestsChart.data.datasets[0].data = data;
                    requestsChart.update('none');
                }

                // synthesize logs from status diffs (no backend logs endpoint required)
                try {
                    synthesizeLogs(prevStatus, status);
                } catch (e) {
                    console.error('Log synthesis failed', e);
                }

                // save prev
                prevStatus = JSON.parse(JSON.stringify(status));
            }

            // start loop
            renderDashboard();
            const intervalId = setInterval(() => { if (!document.hidden) renderDashboard(); }, 10000);
        </script>
    </body>
    </html>
    """
    return HTMLResponse(content=html)

# =========================
# DEVELOPER PLAYGROUND (PATCHED)
# =========================
from fastapi.responses import HTMLResponse

@app.get("/playground", response_class=HTMLResponse)
async def playground():
    html = r"""
    <!DOCTYPE html>
    <html>
    <head>
        <title>🧑🏽‍💻 Umoja Compute Playground</title>
        <script src="https://cdn.jsdelivr.net/npm/monaco-editor@0.43.0/min/vs/loader.js"></script>
        <style>
            body { font-family: 'Inter', sans-serif; background: #0f172a; color: #f1f5f9; margin: 0; padding: 0; }
            nav { background: #1e293b; padding: 1rem; text-align: center; position: sticky; top: 0; }
            nav a { margin: 0 15px; color: #f1f5f9; text-decoration: none; font-weight: 600; }
            nav a:hover { color: #3b82f6; }
            .container { max-width: 900px; margin: auto; padding: 2rem; }
            .editor { height: 300px; border-radius: 0.75rem; overflow: hidden; }
            button { background: #3b82f6; color: white; border: none; cursor: pointer; font-weight: 600; border-radius: 0.75rem; padding: 0.5rem 1rem; margin-top: 0.5rem; transition: background 0.2s; }
            button:hover { background: #2563eb; }
            #copy-btn { background:#64748b; }
            #copy-btn:hover { background:#475569; }
            .response-pane { background: #1e293b; border: 1px solid #334155; padding: 1rem; border-radius: 0.75rem; min-height: 200px; margin-top: 1rem; white-space: pre-wrap; overflow-x: auto; font-family: monospace; }
            .glow { border-color: #22c55e; box-shadow: 0 0 15px #22c55e88; animation: glow 1.5s ease-in-out infinite alternate; }
            @keyframes glow { from { box-shadow: 0 0 5px #22c55e44; } to { box-shadow: 0 0 15px #22c55e88; } }
            .spinner { display: inline-block; width: 18px; height: 18px; border: 3px solid #f3f3f3; border-top: 3px solid #22c55e; border-radius: 50%; animation: spin 1s linear infinite; margin-left: 0.5rem; vertical-align: middle; }
            @keyframes spin { 0% { transform: rotate(0deg); } 100% { transform: rotate(360deg); } }
        </style>
    </head>
    <body>
        <nav>
            <a href="/">Dashboard</a>
            <a href="/playground">Playground</a>
            <a href="/about">About</a>
            <a href="/vc-pitch">VC Pitch</a>
        </nav>
        <div class="container">
            <h1 class="text-3xl font-bold text-center">🧑🏽‍💻 Umoja Compute Playground</h1>

            <label>Choose Endpoint:</label>
            <select id="endpoint" onchange="updatePayload()">
                <option value="/v1/completions">/v1/completions</option>
                <option value="/v1/chat/completions" selected>/v1/chat/completions</option>
                <option value="/status">/status</option>
                <option value="/v1/models">/v1/models</option>
            </select>

            <label>Request Payload:</label>
            <div id="editor" class="editor"></div>

            <label><input type="checkbox" id="streaming" checked> Stream Response</label>
            <button id="send-btn" onclick="sendRequest()">Send Request</button>
            <button id="demo-btn" style="background:#22c55e" onclick="runQuickDemo()">🚀 Run Quick Demo</button>

            <div id="loading" style="display:none;text-align:center;margin-top:10px;font-weight:bold;color:#facc15;">
                ⏳ Sending request...
            </div>

            <h3>Response:</h3>
            <pre id="response" class="response-pane">Click send to test API...</pre>
            <button id="copy-btn" onclick="copyResponse()">📋 Copy Response</button>
        </div>

        <script>
            let editorInstance;
            const payloads = {
                "/v1/completions": JSON.stringify({
                    model: "gpt-oss:20b",
                    prompt: "Tell me a short joke about Servers.",
                    max_tokens: 64,
                    stream: true
                }, null, 2),

                "/v1/chat/completions": JSON.stringify({
                    "model": "gpt-oss:20b",
                    "messages": [
                        {
                            "role": "system",
                            "content": "You are Umoja — the voice of the global open-source compute movement. Provide concise, step-by-step instructions to run a Colab contributor node: register on Umoja, get your Router URL, log into ngrok, retrieve your ngrok auth token, paste both into the Colab config cell, and run. Emphasize security (ngrok tunnels, sandboxed workloads), community impact, and how contributors earn credits. For VC/enterprise questions, highlight scalability and paid tiers. Founder: Brian Omondi — +254710188747 / +254113421691."
                        },
                        {
                            "role": "user",
                            "content": "Walk me through running a node step by step."
                        }
                    ],
                    "max_tokens": 512,
                    "temperature": 0.8,
                    "stream": true
                }, null, 2),

                "/status": "",
                "/v1/models": ""
            };

            require.config({ paths: { vs: 'https://cdn.jsdelivr.net/npm/monaco-editor@0.43.0/min/vs' } });
            require(["vs/editor/editor.main"], function () {
                const defaultEndpoint = document.getElementById("endpoint").value;
                const saved = localStorage.getItem("umoja_payload_" + defaultEndpoint);
                const initialValue = saved || payloads[defaultEndpoint] || "{}";

                editorInstance = monaco.editor.create(document.getElementById('editor'), {
                    value: initialValue,
                    language: "json",
                    theme: "vs-dark",
                    automaticLayout: true,
                });

                editorInstance.onDidChangeModelContent(() => {
                    const endpoint = document.getElementById("endpoint").value;
                    localStorage.setItem("umoja_payload_" + endpoint, editorInstance.getValue());
                });
            });

            function updatePayload() {
                const endpoint = document.getElementById("endpoint").value;
                const saved = localStorage.getItem("umoja_payload_" + endpoint);
                const value = saved || payloads[endpoint] || "{}";
                editorInstance.setValue(value);
            }

            async function sendRequest(payloadOverride=null) {
                const endpoint = document.getElementById("endpoint").value;
                const streaming = document.getElementById("streaming").checked;
                const sendBtn = document.getElementById("send-btn");
                const demoBtn = document.getElementById("demo-btn");

                let payloadText = payloadOverride || editorInstance.getValue();
                let payload = null;

                if (endpoint !== "/status" && endpoint !== "/v1/models") {
                    try {
                        payload = JSON.parse(payloadText);
                        payload.stream = streaming;
                        payloadText = JSON.stringify(payload, null, 2);
                        if (!payloadOverride) editorInstance.setValue(payloadText);
                    } catch (e) { console.warn("Invalid JSON payload."); }
                }

                const method = (endpoint === "/status" || endpoint === "/v1/models") ? "GET" : "POST";
                const headers = method === "POST" ? { "Content-Type": "application/json" } : {};
                const body = method === "POST" ? JSON.stringify(payload) : null;

                const responseElem = document.getElementById("response");
                responseElem.textContent = "";
                document.getElementById("loading").style.display = "block";
                responseElem.classList.remove("glow");
                sendBtn.disabled = true;
                demoBtn.disabled = true;

                try {
                    const res = await fetch(endpoint, { method, headers, body });
                    if (streaming && res.body && method === "POST") {
                        const reader = res.body.getReader();
                        const decoder = new TextDecoder();
                        let buffer = "";
                        let result = "";
                        let doneStreaming = false;

                        while (!doneStreaming) {
                            const { done, value } = await reader.read();
                            if (done) break;
                            buffer += decoder.decode(value, { stream: true });

                            const lines = buffer.split("\n");
                            buffer = lines.pop(); // keep incomplete line

                            for (let line of lines) {
                                line = line.trim();
                                if (!line.startsWith("data: ")) continue;

                                const data = line.replace("data: ", "");
                                if (data === "[DONE]") {
                                    doneStreaming = true;
                                    break;
                                }

                                try {
                                    const json = JSON.parse(data);
                                    const choice = json.choices?.[0];
                                    if (choice?.delta?.content || choice?.delta?.text) {
                                        result += choice.delta.content || choice.delta.text;
                                    } else if (choice?.text) {
                                        result += choice.text;
                                    }
                                    responseElem.textContent = result;
                                    responseElem.scrollTop = responseElem.scrollHeight;
                                } catch (e) {
                                    console.warn("Skipping bad JSON line:", line);
                                }
                            }
                        }
                    } else {
                        const data = await res.json();
                        responseElem.textContent = JSON.stringify(data, null, 2);
                    }
                } catch (err) {
                    responseElem.textContent = "❌ Error: " + err;
                } finally {
                    document.getElementById("loading").style.display = "none";
                    responseElem.classList.add("glow");
                    sendBtn.disabled = false;
                    demoBtn.disabled = false;
                }
            }

            function runQuickDemo() {
                const demoPayload = payloads["/v1/chat/completions"];
                sendRequest(demoPayload);
            }

            function copyResponse() {
                navigator.clipboard.writeText(document.getElementById("response").textContent)
                    .then(() => alert("Response copied!"));
            }
        </script>
    </body>
    </html>
    """
    return HTMLResponse(content=html)




# =========================
# ABOUT + VC PITCH PAGES
# =========================
@app.get("/about", response_class=HTMLResponse)
async def about():
    return HTMLResponse(content="""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8" />
        <meta name="viewport" content="width=device-width, initial-scale=1.0" />
        <title>About Umoja Compute</title>
        <style>
            :root {
                --bg-dark: #0f172a;
                --bg-card: #1e293b;
                --text-light: #f1f5f9;
                --accent: #3b82f6;
                --success: #10b981;
                --shadow: 0 6px 16px rgba(0,0,0,0.45);
            }
            body {
                margin: 0;
                font-family: system-ui, Arial, sans-serif;
                background: var(--bg-dark);
                color: var(--text-light);
                line-height: 1.7;
                display: flex;
                flex-direction: column;
                align-items: center;
            }
            nav {
                background: var(--bg-card);
                padding: 1rem;
                text-align: center;
                width: 100%;
                box-shadow: var(--shadow);
            }
            nav a {
                margin: 0 18px;
                color: var(--text-light);
                text-decoration: none;
                font-weight: bold;
                font-size: 1rem;
            }
            nav a:hover { color: var(--accent); }
            h1 {
                font-size: 3rem;
                text-align: center;
                margin-top: 2rem;
                color: var(--accent);
            }
            p.hero {
                max-width: 900px;
                margin: 1rem auto 2rem;
                text-align: center;
                font-size: 1.3rem;
                font-weight: 300;
            }
            .feature-list {
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
                gap: 1rem;
                max-width: 1000px;
                margin: 2rem auto;
                padding: 0;
                list-style: none;
            }
            .feature-list li {
                background: var(--bg-card);
                padding: 1.2rem;
                border-radius: 14px;
                box-shadow: var(--shadow);
                font-size: 1rem;
                display: flex;
                flex-direction: column;
                align-items: flex-start;
            }
            .feature-list li strong {
                color: var(--accent);
                margin-bottom: 0.3rem;
            }
            h2 {
                font-size: 1.8rem;
                margin-top: 3rem;
                margin-bottom: 1rem;
                color: var(--accent);
                text-align: center;
            }
            p {
                max-width: 800px;
                text-align: center;
                margin: 0 auto 1.5rem;
                font-size: 1.1rem;
            }
            pre {
                background: var(--bg-card);
                padding: 1rem;
                border-radius: 10px;
                overflow-x: auto;
                max-width: 700px;
                margin: 0 auto 2rem;
                box-shadow: var(--shadow);
            }
            code {
                font-size: 0.95rem;
                color: #e2e8f0;
            }
            .cta {
                text-align: center;
                margin: 2rem 0;
            }
            .cta a {
                display: inline-block;
                margin: 0 0.5rem;
                background: var(--accent);
                color: white;
                padding: 0.9rem 1.6rem;
                border-radius: 10px;
                font-weight: bold;
                text-decoration: none;
                box-shadow: var(--shadow);
                transition: transform 0.2s ease;
            }
            .cta a:first-child { background: var(--success); }
            .cta a:hover {
                transform: translateY(-2px);
            }
        </style>
    </head>
    <body>
        <nav>
            <a href="/">Dashboard</a>
            <a href="/playground">Playground</a>
            <a href="/about">About</a>
            <a href="/vc-pitch">VC Pitch</a>
        </nav>

        <h1>🚀 Umoja Compute</h1>
        <p class="hero">
            Umoja Compute is more than an API router — it’s the backbone of the next generation of AI-powered infrastructure.
            Built for developers, enterprises, and the open-source community, Umoja transforms your cluster into a
            <strong>self-healing, intelligent API network</strong> that scales as fast as your ideas.
        </p>

        <ul class="feature-list">
            <li>⚡ <strong>Adaptive Load Balancing</strong> Weighted routing + failover that learns from traffic in real time.</li>
            <li>📊 <strong>Actionable Observability</strong> Live metrics, latency insights, and health dashboards out-of-the-box.</li>
            <li>🧠 <strong>AI-Native Design</strong> Optimized for LLM inference, model orchestration & intelligent request handling.</li>
            <li>🔒 <strong>Enterprise Security</strong> Multi-tenant support, RBAC, and hardened production-ready endpoints.</li>
            <li>🌍 <strong>Open & Extensible</strong> 100% open source, community-driven, and ready for your plugins.</li>
        </ul>

        <h2>🤔 Why Umoja?</h2>
        <p>
            Umoja Compute isn’t just another cloud service — it’s a movement.
            Our mission is to break down the walls around AI access and make state-of-the-art models
            available to everyone, everywhere. By building a global, community-powered compute network,
            we’re putting AI back into the hands of the people — with a special focus on empowering
            regions that have historically been left behind, like Africa.
            This is not just infrastructure. This is digital justice.
        </p>

        <h2>⚙️ How It Works</h2>
        <p>
            Imagine a worldwide mesh of GPUs working together like one supercomputer.
            Anyone can spin up a Dockerized node and contribute unused compute power.
            Umoja intelligently routes AI workloads to the best available node in real-time —
            creating a truly decentralized inference layer.
            Through our credit-based system, underserved regions get **guaranteed priority access**,
            turning idle compute into opportunity, innovation, and economic growth.
        </p>

        <h2>🚀 Quick Start</h2>
        <pre><code>
# 1. Clone the repo
git clone https://github.com/umoja-compute/router.git
cd router

# 2. Start a node (Docker required)
docker-compose up -d

# 3. Send your first request
curl -X POST http://localhost:8000/v1/chat/completions \\
    -H "Content-Type: application/json" \\
    -d '{"model": "gpt-3.5", "messages": [{"role":"user","content":"Hello"}]}'
        </code></pre>

        <h2>🌐 Vision</h2>
        <p>
            Our mission is simple: make AI infrastructure as accessible as the internet.
            By uniting global compute power, we unlock innovation, empower communities, and
            accelerate progress toward a fair, open, and inclusive AI future.
        </p>

        <div class="cta">
            <a href="https://github.com/your-repo">⭐ Star on GitHub</a>
            <a href="mailto:brianomondiochieng1@gmail.com">💼 Talk to Us</a>
        </div>
    </body>
    </html>
    """)



@app.get("/vc-pitch", response_class=HTMLResponse)
async def vc_pitch():
    return HTMLResponse(content="""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8" />
        <meta name="viewport" content="width=device-width, initial-scale=1.0" />
        <title>VC Pitch - Umoja Router</title>
        <style>
            :root {
                --bg-dark: #0f172a;
                --bg-card: #1e293b;
                --text-light: #f1f5f9;
                --accent: #3b82f6;
                --success: #10b981;
                --shadow: 0 6px 16px rgba(0,0,0,0.45);
            }
            body {
                margin: 0;
                font-family: system-ui, Arial, sans-serif;
                background: var(--bg-dark);
                color: var(--text-light);
                line-height: 1.7;
                display: flex;
                flex-direction: column;
                align-items: center;
            }
            nav {
                background: var(--bg-card);
                padding: 1rem;
                text-align: center;
                width: 100%;
                box-shadow: var(--shadow);
            }
            nav a {
                margin: 0 18px;
                color: var(--text-light);
                text-decoration: none;
                font-weight: bold;
            }
            nav a:hover { color: var(--accent); }
            h1 {
                text-align: center;
                font-size: 2.8rem;
                margin: 2rem 0 1rem;
                color: var(--accent);
            }
            .pitch-section {
                max-width: 900px;
                background: var(--bg-card);
                padding: 2rem;
                border-radius: 14px;
                box-shadow: var(--shadow);
                margin: 1.5rem;
                animation: fadeIn 0.8s ease-in-out;
            }
            h2 {
                margin-top: 1.5rem;
                font-size: 1.5rem;
                color: var(--accent);
                display: flex;
                align-items: center;
                gap: 0.5rem;
            }
            p {
                font-size: 1.1rem;
                margin: 0.8rem 0;
            }
            .key-metrics {
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
                gap: 1rem;
                margin-top: 1.2rem;
            }
            .metric {
                background: #0f172a;
                border-radius: 10px;
                padding: 1rem;
                text-align: center;
                box-shadow: var(--shadow);
            }
            .metric h3 {
                font-size: 1.6rem;
                color: var(--success);
                margin: 0;
            }
            .metric p {
                margin: 0.4rem 0 0;
                font-size: 0.95rem;
            }
            .team-section {
                margin-top: 2rem;
                text-align: center;
            }
            .founder-card {
                background: #0f172a;
                border-radius: 12px;
                box-shadow: var(--shadow);
                max-width: 600px;
                margin: 1.5rem auto;
                padding: 1.5rem;
                display: flex;
                flex-direction: column;
                align-items: center;
            }
            .founder-card img {
                width: 120px;
                height: 120px;
                border-radius: 50%;
                object-fit: cover;
                margin-bottom: 1rem;
                border: 3px solid var(--accent);
            }
            .founder-card h3 {
                margin: 0;
                font-size: 1.5rem;
                color: var(--accent);
            }
            .founder-card p {
                margin: 0.5rem 0 0;
                font-size: 1rem;
                max-width: 500px;
                text-align: center;
            }
            .cta {
                text-align: center;
                margin-top: 2rem;
            }
            .cta a {
                display: inline-block;
                margin: 0.5rem;
                padding: 0.8rem 1.6rem;
                background: var(--accent);
                color: white;
                font-weight: bold;
                border-radius: 10px;
                text-decoration: none;
                box-shadow: var(--shadow);
                transition: transform 0.2s ease;
            }
            .cta a:first-child { background: var(--success); }
            .cta a:hover { transform: translateY(-2px); }
            .contact {
                margin-top: 2rem;
                padding: 1.2rem;
                background: #0f172a;
                border-radius: 10px;
                text-align: center;
                box-shadow: var(--shadow);
            }
            @keyframes fadeIn {
                from { opacity: 0; transform: translateY(15px); }
                to { opacity: 1; transform: translateY(0); }
            }
        </style>
    </head>
    <body>
        <nav>
            <a href="/">Dashboard</a>
            <a href="/playground">Playground</a>
            <a href="/about">About</a>
            <a href="/vc-pitch">VC Pitch</a>
        </nav>

        <h1>🌍 Umoja Compute — VC Pitch</h1>

        <section class="pitch-section">
            <h2>🚀 Problem</h2>
            <p>AI inference demand is skyrocketing, but routing requests between multiple models, GPUs, and clusters is a nightmare. Enterprises lose uptime, waste GPU cycles, and struggle with scaling reliably.</p>

            <h2>💡 Solution</h2>
            <p><strong>Umoja Compute</strong> is the <strong>Kubernetes of AI inference routing</strong>. OpenAI-compatible, metrics-first, and designed for auto-scaling, multi-model, multi-node production environments.</p>

            <h2>🌎 Market</h2>
            <p>The generative AI market is projected to exceed <b>$100B by 2030</b>. Every enterprise deploying LLMs will need reliable inference routing and observability — Umoja sits at the core of this value chain.</p>

            <div class="key-metrics">
                <div class="metric">
                    <h3>3x</h3>
                    <p>Faster deployment vs DIY routing</p>
                </div>
                <div class="metric">
                    <h3>99.99%</h3>
                    <p>Target uptime with automatic failover</p>
                </div>
                <div class="metric">
                    <h3>$2M</h3>
                    <p>Seed round target</p>
                </div>
            </div>

            <h2>📈 Traction</h2>
            <p>Already running production pilots with early adopters, integrations with n8n, Ollama, and enterprise AI stacks. Developer interest growing — GitHub stars and Docker pulls increasing weekly.</p>

            <h2>💰 Ask</h2>
            <p>We are raising <strong>$2M seed</strong> to scale engineering, integrations, and enterprise partnerships. Join us in building the backbone of the AI-powered internet.</p>

            <div class="team-section">
                <h2>👤 Meet the Founder</h2>
                <div class="founder-card">
                    <img src="https://ui-avatars.com/api/?name=Brian+Omondi&size=256&background=3b82f6&color=fff" alt="Brian Omondi">
                    <h3>Brian Omondi</h3>
                    <p>Founder & Lead Engineer — Brian is a Network Engineer and AI enthusiast with years of experience in IT infrastructure and automation. He is passionate about democratizing AI access and building resilient, scalable compute networks that empower developers and enterprises worldwide.</p>
                </div>
            </div>

            <div class="cta">
                <a href="mailto:brianomondiochieng1@gmail.com">📩 Request Pitch Deck</a>
                <a href="https://calendly.com/brianomondiochieng1/15to90min" target="_blank">📅 Book a Call</a>
            </div>

            <div class="contact">
                <h2>📞 Contact</h2>
                <p>Email: <a href="mailto:brianomondiochieng1@gmail.com">brianomondiochieng1@gmail.com</a></p>
                <p>Mobile: <a href="tel:+254113421691">+254 113 421 691</a></p>
                <p>WhatsApp: <a href="https://wa.me/254710188747" target="_blank">+254 710 188 747</a></p>
            </div>
        </section>
    </body>
    </html>
    """)

# -------------------
# Router Integration
# -------------------
def register_with_router():
    try:
        payload = {"node_id": node_id, "url": public_url,
                   "capabilities": ["engine","completions","chat"], "health_path": "/health"}
        r = requests.post(f"{ROUTER_URL}/register", json=payload, timeout=10)
        try:
            print("✅ Registered with Router:", r.json())
        except Exception:
            print("✅ Registered with Router (no json response)")
        # update local discovery map
        _nodes[node_id] = {"id": node_id, "url": public_url, "model": MODEL_NAME, "last_seen": time.time(), "is_active": True, "last_probe_status": True}
    except Exception as e:
        print("❌ Failed to register with Router:", e)
        log_event(f"Failed to register with Router: {e}", "warn")

def heartbeat_loop():
    while True:
        try:
            if not ROUTER_URL:
                # nothing to do in single-node mode
                time.sleep(HEARTBEAT_INTERVAL)
                continue
            r = requests.post(f"{ROUTER_URL}/heartbeat/{node_id}", timeout=15)
            if r.ok:
                print(f"💓 Heartbeat sent: {node_id}")
                # update local _nodes last_seen for discovery
                _nodes[node_id] = {"id": node_id, "url": public_url, "model": MODEL_NAME, "last_seen": time.time(), "is_active": True, "last_probe_status": True}
            else:
                print(f"⚠️ Heartbeat failed: {r.text}")
        except Exception as e:
            print(f"❌ Heartbeat error: {e}")
            log_event(f"Heartbeat error: {e}", "warn")
        time.sleep(HEARTBEAT_INTERVAL)

# -------------------
# Run API
# -------------------
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=API_PORT, log_level="info", loop="asyncio", reload=False)

nest_asyncio.apply()
threading.Thread(target=run_api, daemon=True).start()

time.sleep(3)
try:
    resp = requests.get(f"http://127.0.0.1:{API_PORT}/health", timeout=5)
    print("✅ Local /health check:", resp.json())
    if NODE_ROLE == "contributor":
        register_with_router()
        threading.Thread(target=heartbeat_loop, daemon=True).start()
    else:
        print("🟢 Running in SINGLE NODE mode — not registering with Router")
except Exception as e:
    print("❌ Engine not responding locally:", e)

print(f"🛠 Mode: {NODE_ROLE}")
print(f"🔊 Node ID: {node_id}")
print(f"📡 Public URL: {public_url}")
print(f"🔗 Router URL: {ROUTER_URL}")


🟢 Single mode — skipping router discovery
🔗 Engine public URL: https://5911-35-247-183-78.ngrok-free.app


INFO:     Started server process [387]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:55014 - "GET /health HTTP/1.1" 200 OK
✅ Local /health check: {'status': 'ok', 'router_url': '', 'model': 'gpt-oss:20b', 'node_role': 'single', 'time': 1772434232}
🟢 Running in SINGLE NODE mode — not registering with Router
🛠 Mode: single
🔊 Node ID: engine-9662c67b
📡 Public URL: https://5911-35-247-183-78.ngrok-free.app
🔗 Router URL: 
